# 📗 Logistic Regression — Applied on Kaggle Titanic Dataset
### Atlas Skilltech University | Instructor Reference

---

**Dataset:** Titanic — Will This Passenger Survive? (Kaggle Classic)
- We will **download it directly** using the `seaborn` built-in datasets (same data, no Kaggle account needed)
- Alternatively, download from: https://www.kaggle.com/c/titanic

**Goal:** Predict whether a passenger **survived (1)** or **died (0)** using features like age, class, sex, etc.

**Structure (follows same order as Notebook 1):**
1. Load Dataset
2. EDA (Exploratory Data Analysis)
3. Preprocessing (handle missing values, encode categories)
4. Train-Test Split + Scaling
5. Model Training
6. Confusion Matrix & Accuracy
7. Precision, Recall, F1 Score
8. ROC Curve & AUC
9. Performance Summary
10. Multiclass Extension (Passenger Class Prediction)
11. Capstone Exercise

> 💡 Every hard keyword is explained in a markdown cell **before** the code.

---
## 🔧 Step 0 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay, roc_curve
)

print('All libraries imported successfully ✅')

---
## 📌 Step 1 — Load the Dataset

**Keywords to know:**
- **seaborn** — has some free built-in datasets we can load with one line
- **Titanic dataset** — 891 passengers, tells us who survived the Titanic shipwreck in 1912

**Column descriptions:**
| Column | Description |
|---|---|
| survived | 0 = No, 1 = Yes (TARGET) |
| pclass | Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd |
| sex | Gender of passenger |
| age | Age in years |
| sibsp | # of siblings/spouses aboard |
| parch | # of parents/children aboard |
| fare | Passenger fare (ticket price) |
| embarked | Port of embarkation (C/Q/S) |
| class | Ticket class as text |
| who | man/woman/child |
| adult_male | True/False |
| deck | Deck of cabin |
| embark_town | City of embarkation |
| alive | yes/no |
| alone | True if no family aboard |

In [ ]:
# Load the Titanic dataset from seaborn (same as Kaggle version)
df = sns.load_dataset('titanic')

print('Shape:', df.shape)
df.head()

In [ ]:
print('Column names:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)

---
## 📌 Step 2 — EDA (Exploratory Data Analysis)

**Keywords to know:**
- **EDA** — looking at the data with charts and statistics to understand patterns before building a model
- **Missing values (NaN)** — blank cells; most ML models can't handle them directly
- **isnull().sum()** — counts how many missing values are in each column
- **value_counts()** — shows how many times each unique value appears

In [ ]:
# Basic statistics
print('Basic Statistics:')
df.describe()

In [ ]:
# Missing values check
print('Missing Values per column:')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Class balance check — always check this for classification!
print('Survival counts:')
print(df['survived'].value_counts())
print()
print('Survival rate:', df['survived'].mean().round(3))

In [ ]:
# EDA Charts — 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# 1. Survival count
df['survived'].value_counts().plot(kind='bar', ax=axes[0,0], color=['salmon', 'steelblue'], edgecolor='black')
axes[0,0].set_title('Survival Count')
axes[0,0].set_xticklabels(['Died (0)', 'Survived (1)'], rotation=0)
axes[0,0].set_ylabel('Count')

# 2. Survival by Sex
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0,1], color=['steelblue', 'salmon'], edgecolor='black')
axes[0,1].set_title('Survival Rate by Sex')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].set_xticklabels(['Female', 'Male'], rotation=0)

# 3. Survival by Passenger Class
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[0,2], color=['gold', 'silver', 'brown'], edgecolor='black')
axes[0,2].set_title('Survival Rate by Passenger Class')
axes[0,2].set_ylabel('Survival Rate')
axes[0,2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)

# 4. Age distribution by survival
df[df['survived']==1]['age'].dropna().hist(bins=20, ax=axes[1,0], alpha=0.7, color='steelblue', label='Survived')
df[df['survived']==0]['age'].dropna().hist(bins=20, ax=axes[1,0], alpha=0.7, color='salmon', label='Died')
axes[1,0].set_title('Age Distribution by Survival')
axes[1,0].set_xlabel('Age')
axes[1,0].legend()

# 5. Fare distribution by survival
df[df['survived']==1]['fare'].hist(bins=20, ax=axes[1,1], alpha=0.7, color='steelblue', label='Survived')
df[df['survived']==0]['fare'].hist(bins=20, ax=axes[1,1], alpha=0.7, color='salmon', label='Died')
axes[1,1].set_title('Fare Distribution by Survival')
axes[1,1].set_xlabel('Fare')
axes[1,1].legend()

# 6. Survival by embarkation point
df.groupby('embark_town')['survived'].mean().plot(kind='bar', ax=axes[1,2], color='teal', edgecolor='black')
axes[1,2].set_title('Survival Rate by Embarkation Town')
axes[1,2].set_ylabel('Survival Rate')
axes[1,2].tick_params(axis='x', rotation=15)

plt.suptitle('Titanic EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — only for numeric columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

---
## 📌 Step 3 — Preprocessing

**Keywords to know:**
- **Imputation** — filling missing values with a sensible substitute (mean, median, mode)
- **Median imputation** — fill missing values with the median (middle value); better than mean when there are outliers
- **Label Encoding** — converts text categories to numbers (e.g., male=1, female=0)
- **Feature selection** — choosing which columns to use; dropping irrelevant or redundant ones
- **drop()** — remove a column from the DataFrame

In [ ]:
# Select useful columns
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target   = 'survived'

df_clean = df[features + [target]].copy()

print('Before cleaning:')
print(df_clean.isnull().sum())

In [ ]:
# Fill missing 'age' with median
df_clean['age'].fillna(df_clean['age'].median(), inplace=True)

# Fill missing 'embarked' with most common value (mode)
df_clean['embarked'].fillna(df_clean['embarked'].mode()[0], inplace=True)

print('After cleaning — missing values:')
print(df_clean.isnull().sum())

In [ ]:
# Encode categorical columns
# sex: female=0, male=1
df_clean['sex'] = LabelEncoder().fit_transform(df_clean['sex'])

# embarked: C=0, Q=1, S=2
df_clean['embarked'] = LabelEncoder().fit_transform(df_clean['embarked'])

print('After encoding:')
print(df_clean.head())
print()
print('Data types:')
print(df_clean.dtypes)

---
## 📌 Step 4 — Train-Test Split + Feature Scaling

**Keywords to know:**
- **X** — feature matrix (all input columns)
- **y** — target vector (what we want to predict)
- **StandardScaler** — transforms features to have mean=0 and standard deviation=1
- **fit_transform** — learn the scaling parameters AND apply them (use only on training data)
- **transform** — apply the already-learned parameters (use on test data)

In [ ]:
X = df_clean.drop('survived', axis=1)
y = df_clean['survived']

print('Feature columns:', X.columns.tolist())
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y → keeps the same class balance in both train and test sets

print(f'Train size : {len(X_train)}')
print(f'Test  size : {len(X_test)}')
print()
print('Train class balance:')
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling done ✅')

---
## 📌 Step 5 — Model Training

**Keywords to know:**
- **solver** — the algorithm used to find the best weights; `lbfgs` is the default for small datasets
- **max_iter** — maximum number of iterations for the solver to find the solution
- **C (regularisation)** — controls model complexity; lower C = more regularisation = simpler model
- **Regularisation** — a technique to prevent overfitting by penalising large coefficients

In [ ]:
model = LogisticRegression(solver='lbfgs', max_iter=1000, C=1.0, random_state=42)
model.fit(X_train_sc, y_train)

print('Model trained ✅')

# Show feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0].round(4)
}).sort_values('Coefficient', key=abs, ascending=False)

print('\nFeature Coefficients (larger absolute value = more influential):')
print(coef_df)

In [ ]:
# Visualise feature coefficients
plt.figure(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'salmon' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression — Feature Coefficients\n(positive = increases survival probability)')
plt.tight_layout()
plt.show()

---
## 📌 Step 6 — Confusion Matrix & Accuracy

**Reminder of keywords:**
- **TP (True Positive)** — predicted Survived, actually Survived
- **TN (True Negative)** — predicted Died, actually Died
- **FP (False Positive / Type I Error)** — predicted Survived, actually Died
- **FN (False Negative / Type II Error)** — predicted Died, actually Survived
- **Accuracy** = (TP + TN) / Total

In [ ]:
y_pred      = model.predict(X_test_sc)
y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f'Accuracy: {acc:.4f} ({acc:.2%})')
print('\nConfusion Matrix (raw):')
print(cm)

In [ ]:
# --- METHOD 1 (older): seaborn heatmap ---
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Died', 'Survived'],
            yticklabels=['Died', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — seaborn heatmap (older style)')
plt.tight_layout()
plt.show()

In [ ]:
# --- METHOD 2 (modern, recommended): ConfusionMatrixDisplay ---
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Died', 'Survived'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — ConfusionMatrixDisplay (modern style)')
plt.tight_layout()
plt.show()

---
## 📌 Step 7 — Precision, Recall, F1 Score

**Recall this:**
$$\text{Precision} = \frac{TP}{TP + FP}$$
$$\text{Recall} = \frac{TP}{TP + FN}$$
$$F_1 = 2 \times \frac{P \times R}{P + R}$$

In the Titanic context:
- **Precision** — of all predicted survivors, how many actually survived?
- **Recall** — of all actual survivors, how many did we correctly identify?

In [ ]:
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.4f}')
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

---
## 📌 Step 8 — ROC Curve & AUC

**Recall this:**
- **ROC Curve** — shows the trade-off between True Positive Rate (sensitivity) and False Positive Rate at every threshold
- **AUC** — Area under the ROC Curve; 1.0 = perfect, 0.5 = random
- We'll show both older (manual) and newer (RocCurveDisplay) approach

In [ ]:
auc = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

print(f'AUC Score: {auc:.4f}')

In [ ]:
# METHOD 1 — Manual ROC curve
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Logistic Reg (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Titanic Survival (Manual)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# METHOD 2 — Modern RocCurveDisplay
RocCurveDisplay.from_estimator(model, X_test_sc, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.title('ROC Curve — Titanic Survival (RocCurveDisplay)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📌 Step 9 — Full Performance Evaluation Summary

**Keywords to know:**
- **Specificity** = TN / (TN + FP) — how well the model identifies deaths
- **Balanced Accuracy** — average of Sensitivity (Recall) and Specificity
- **Log Loss** — the actual value being minimised during training

In [ ]:
from sklearn.metrics import log_loss, balanced_accuracy_score

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print('=' * 50)
print('     TITANIC LOGISTIC REGRESSION PERFORMANCE')
print('=' * 50)
print(f'  True Positives  (TP) : {tp}')
print(f'  True Negatives  (TN) : {tn}')
print(f'  False Positives (FP) : {fp}')
print(f'  False Negatives (FN) : {fn}')
print('-' * 50)
print(f'  Accuracy         : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision        : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall (Sensitivity): {recall_score(y_test, y_pred):.4f}')
print(f'  Specificity      : {specificity:.4f}')
print(f'  F1 Score         : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC          : {auc:.4f}')
print(f'  Log Loss         : {log_loss(y_test, y_pred_prob):.4f}')
print('=' * 50)

---
## 📌 Step 10 — Multiclass: Predicting Passenger Class (pclass)

**Keywords to know:**
- We now predict `pclass` (1st, 2nd, 3rd class) — a 3-class problem
- **multi_class='multinomial'** — uses softmax to output probabilities across all 3 classes
- **macro avg** — treats all classes equally regardless of size
- **weighted avg** — accounts for class imbalance

In [ ]:
# Features for predicting passenger class
features_mc = ['sex', 'age', 'sibsp', 'parch', 'fare', 'survived']
target_mc   = 'pclass'

df_mc = df[features_mc + [target_mc]].copy()
df_mc['age'].fillna(df_mc['age'].median(), inplace=True)
df_mc['sex'] = LabelEncoder().fit_transform(df_mc['sex'])

X_mc = df_mc.drop('pclass', axis=1)
y_mc = df_mc['pclass']

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc
)

sc_mc = StandardScaler()
X_train_mc_sc = sc_mc.fit_transform(X_train_mc)
X_test_mc_sc  = sc_mc.transform(X_test_mc)

multi_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
multi_model.fit(X_train_mc_sc, y_train_mc)

y_pred_mc = multi_model.predict(X_test_mc_sc)

print('Multiclass Classification Report — Predicting Passenger Class:')
print(classification_report(y_test_mc, y_pred_mc, target_names=['1st Class', '2nd Class', '3rd Class']))

In [ ]:
# Confusion matrix for 3-class
ConfusionMatrixDisplay.from_estimator(
    multi_model, X_test_mc_sc, y_test_mc,
    display_labels=['1st Class', '2nd Class', '3rd Class'],
    cmap='Blues'
)
plt.title('Multiclass Confusion Matrix — Passenger Class Prediction')
plt.tight_layout()
plt.show()

---
## 🏁 Step 11 — Capstone Exercise

### Challenge for Students

Using the Titanic dataset, build the **best possible** logistic regression model.

**Tasks:**
1. Create a new feature `family_size = sibsp + parch + 1`
2. Create a feature `is_alone = 1 if family_size == 1 else 0`
3. Try different values of `C` (e.g., 0.01, 0.1, 1.0, 10) and compare accuracy
4. Report all metrics: Accuracy, Precision, Recall, F1, AUC
5. Plot the confusion matrix and ROC curve

**Template code to get you started:**

In [ ]:
# YOUR CODE HERE — Capstone Solution Template
# Step 1: Feature Engineering
df_cap = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']].copy()
df_cap['age'].fillna(df_cap['age'].median(), inplace=True)
df_cap['embarked'].fillna(df_cap['embarked'].mode()[0], inplace=True)
df_cap['sex'] = LabelEncoder().fit_transform(df_cap['sex'])
df_cap['embarked'] = LabelEncoder().fit_transform(df_cap['embarked'])

# NEW FEATURES
df_cap['family_size'] = df_cap['sibsp'] + df_cap['parch'] + 1
df_cap['is_alone']    = (df_cap['family_size'] == 1).astype(int)

print('New DataFrame head:')
print(df_cap.head())

# Step 2: Train/Test split
X_cap = df_cap.drop('survived', axis=1)
y_cap = df_cap['survived']

X_train_cap, X_test_cap, y_train_cap, y_test_cap = train_test_split(
    X_cap, y_cap, test_size=0.2, random_state=42, stratify=y_cap
)

sc_cap = StandardScaler()
X_train_cap_sc = sc_cap.fit_transform(X_train_cap)
X_test_cap_sc  = sc_cap.transform(X_test_cap)

In [ ]:
# Step 3: Try different C values
print(f'{'C value':>10} | {'Accuracy':>10} | {'F1':>8} | {'AUC':>8}')
print('-' * 45)

for c_val in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(C=c_val, max_iter=1000, random_state=42)
    m.fit(X_train_cap_sc, y_train_cap)
    yp = m.predict(X_test_cap_sc)
    ypp = m.predict_proba(X_test_cap_sc)[:, 1]
    print(f'{c_val:>10} | {accuracy_score(y_test_cap, yp):>10.4f} | {f1_score(y_test_cap, yp):>8.4f} | {roc_auc_score(y_test_cap, ypp):>8.4f}')

In [ ]:
# Step 4: Final best model
best_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
best_model.fit(X_train_cap_sc, y_train_cap)

y_pred_cap  = best_model.predict(X_test_cap_sc)
y_proba_cap = best_model.predict_proba(X_test_cap_sc)[:, 1]

print('=== CAPSTONE FINAL MODEL RESULTS ===')
print(classification_report(y_test_cap, y_pred_cap, target_names=['Died', 'Survived']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm_cap = confusion_matrix(y_test_cap, y_pred_cap)
ConfusionMatrixDisplay(cm_cap, display_labels=['Died', 'Survived']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix — Capstone Model')

# ROC curve
fpr_c, tpr_c, _ = roc_curve(y_test_cap, y_proba_cap)
auc_c = roc_auc_score(y_test_cap, y_proba_cap)
axes[1].plot(fpr_c, tpr_c, color='darkorange', lw=2, label=f'AUC = {auc_c:.2f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — Capstone Model')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## ✅ Summary — What We Applied on Real Data

| Step | What We Did |
|---|---|
| Data Loading | Loaded Titanic dataset (891 rows, real-world messy data) |
| EDA | Visualised survival rates by sex, class, age, fare |
| Preprocessing | Imputed missing age, encoded sex and embarked |
| Train-Test Split | 80/20 split, stratified to preserve class balance |
| Feature Scaling | StandardScaler on both train and test sets |
| Model Training | LogisticRegression with lbfgs solver |
| Confusion Matrix | Both seaborn heatmap AND ConfusionMatrixDisplay |
| Precision/Recall/F1 | Full classification report |
| ROC / AUC | Manual plot AND RocCurveDisplay |
| Multiclass | Predicted passenger class (3 categories) |
| Capstone | Feature engineering + hyperparameter tuning |

---
*Next: Read the Google Doc for theory + formula reference →*